# modeling_v5 — Row-level 예측 (step 단위 → WF 평균)

**방향**: WF 단위 통계 집계(천장 RMSE ~62)에서 벗어나, row(=step) 단위로 C65를 직접 예측한 뒤 WF 내 평균으로 집계.

**세션2 재탐색 반영**
- `C23`(28종 Recipe) 추가 — out-of-fold 타깃인코딩(누수 차단). 이전 버전에서 누락됐던 유일한 실질 신호.
- `C7`(step)을 LightGBM 범주형으로 명시 → step×센서 상호작용 학습 (C17 상관이 step별 0.19~0.80).
- 각 row에 WF 전역 context(주요 센서 WF 집계) broadcast → 로컬(step)+글로벌 동시 사용.
- 제외: `C36`(=C7 완전중복), `C30`(상수), `C40`/`C41`(신호 없음).
- v4의 `.values` 버그 방지: **DataFrame으로 학습(컬럼명 유지)**.
- GroupKFold(C64), OOF는 WF 평균 후 WF-level RMSE로 평가.

**비교 기준**: v3 최선 Valid 62.31 / Test 60.51 → 목표 ~40.

## Cell 1 — imports & 설정

In [1]:
import numpy as np, pandas as pd, time, json, warnings
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
warnings.filterwarnings("ignore")

DATA_DIR = Path("../문제1(하)"); ANS_DIR = Path("../문제1_하_answer")
OUTPUT_DIR = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
WF_ID, TARGET, N_FOLDS = "C64", "C65", 5

# row-level 피처에서 제외할 컬럼
DROP_IDS   = ["C34","C35","C38"]
DROP_LOT   = ["C20","C21","C22"]
DROP_CONST = ["C14","C24","C30"]          # C30 상수 추가
DROP_EXCL  = ["C26","C28","C29","C37"]
DROP_ALLNA = ["C2","C13","C43","C47","C53","C55"]
DROP_TIME  = ["C10","C39","C40","C41"]    # C40/C41 신호 없음
DROP_DUP   = ["C36"]                        # C36 == C7 (완전중복)
DROP_ALL = DROP_IDS+DROP_LOT+DROP_CONST+DROP_EXCL+DROP_ALLNA+DROP_TIME+DROP_DUP
CAT_COLS = ["C6","C7"]      # LightGBM 범주형
TE_COL   = "C23"            # 타깃인코딩 대상

## Cell 2 — 데이터 로드

In [2]:
train  = pd.read_csv(DATA_DIR/"train_data.csv")
valid_X= pd.read_csv(DATA_DIR/"valid_X.csv")
test_X = pd.read_csv(DATA_DIR/"test_X.csv")
valid_Yp = pd.read_csv(DATA_DIR/"valid_Y_problem.csv")
test_Yp  = pd.read_csv(DATA_DIR/"test_Y_problem.csv")
valid_Ya = pd.read_csv(ANS_DIR/"valid_Y_answer.csv").set_index("C64")["C65"]
test_Ya  = pd.read_csv(ANS_DIR/"test_Y_answer.csv").set_index("C64")["C65"]

cols_no_t = [c for c in train.columns if c != TARGET]
valid_X = valid_X[cols_no_t]; test_X = test_X[cols_no_t]
print(train.shape, valid_X.shape, test_X.shape)

(123614, 65) (20577, 64) (20510, 64)


## Cell 3 — row-level 피처 빌더 (WF context broadcast 포함)

In [3]:
def build_rows(df, has_target=True):
    df = df.copy()
    # 수치 센서 (범주/제외/타깃/TE/WF키 제외)
    excl = set([WF_ID,TARGET]+DROP_ALL+CAT_COLS+[TE_COL])
    sensors = [c for c in df.select_dtypes(include=[np.number]).columns if c not in excl]

    # --- WF 전역 context: 주요 센서 WF 집계를 각 row에 broadcast ---
    g = df.groupby(WF_ID)
    ctx = g[sensors].agg(["mean","std","min","max"])
    ctx.columns = [f"{c}_wf_{s}" for c,s in ctx.columns]
    ctx["wf_nrows"] = g.size()
    df = df.merge(ctx, on=WF_ID, how="left")

    # --- step 내 위치 (WF 안에서 몇 번째 row인지) ---
    df["row_pos"] = g.cumcount()

    feat_cols = sensors + list(ctx.columns) + ["row_pos"] + CAT_COLS
    for c in CAT_COLS:
        df[c] = df[c].astype("category")
    out = df[[WF_ID]+feat_cols].copy()
    out[TE_COL] = df[TE_COL].values
    if has_target: out[TARGET] = df[TARGET].values
    return out, feat_cols

tr_rows, FEATS = build_rows(train, True)
va_rows, _     = build_rows(valid_X, False)
te_rows, _     = build_rows(test_X, False)
print("row-level 피처 수:", len(FEATS)+1, "| train rows:", len(tr_rows))

row-level 피처 수: 185 | train rows: 123614


## Cell 4 — C23 out-of-fold 타깃인코딩 헬퍼 (스무딩)

In [4]:
def te_fit(frame, col, target, m=20):
    prior = frame[target].mean()
    agg = frame.groupby(col)[target].agg(["mean","count"])
    enc = (agg["mean"]*agg["count"] + prior*m) / (agg["count"]+m)
    return enc, prior

## Cell 5 — GroupKFold row-level 학습 → WF 평균 집계

In [5]:
PARAMS = dict(objective="regression", metric="rmse", boosting_type="gbdt",
    learning_rate=0.03, num_leaves=127, max_depth=-1, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
    reg_alpha=0.5, reg_lambda=1.0, n_estimators=4000,
    random_state=42, n_jobs=-1, verbose=-1)

y  = tr_rows[TARGET].values
gr = tr_rows[WF_ID].astype("category").cat.codes.values
gkf = GroupKFold(N_FOLDS)

oof = np.zeros(len(tr_rows))
va_pred = np.zeros(len(va_rows)); te_pred = np.zeros(len(te_rows))
USE = FEATS + [TE_COL+"_te"]

for k,(tri,vai) in enumerate(gkf.split(tr_rows, y, gr)):
    tf = tr_rows.iloc[tri].copy(); vf = tr_rows.iloc[vai].copy()
    enc, prior = te_fit(tf, TE_COL, TARGET)
    tf[TE_COL+"_te"] = tf[TE_COL].map(enc).fillna(prior)
    vf[TE_COL+"_te"] = vf[TE_COL].map(enc).fillna(prior)
    va_k = va_rows.copy(); te_k = te_rows.copy()
    va_k[TE_COL+"_te"] = va_k[TE_COL].map(enc).fillna(prior)
    te_k[TE_COL+"_te"] = te_k[TE_COL].map(enc).fillna(prior)

    m = lgb.LGBMRegressor(**PARAMS)
    m.fit(tf[USE], tf[TARGET],
          eval_set=[(vf[USE], vf[TARGET])],
          categorical_feature=CAT_COLS,
          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    oof[vai] = m.predict(vf[USE])
    va_pred += m.predict(va_k[USE]) / N_FOLDS
    te_pred += m.predict(te_k[USE]) / N_FOLDS
    print(f"fold{k+1} best_iter={m.best_iteration_}")

fold1 best_iter=190
fold2 best_iter=264
fold3 best_iter=195
fold4 best_iter=200
fold5 best_iter=168


## Cell 6 — WF 단위 집계 & 평가

In [6]:
def wf_agg(rows, pred):
    return pd.DataFrame({WF_ID:rows[WF_ID].values, "p":pred}).groupby(WF_ID)["p"].mean()

# OOF는 row→WF 평균, 실제 타깃도 WF 단위
oof_wf = wf_agg(tr_rows, oof)
y_wf   = tr_rows.groupby(WF_ID)[TARGET].first()
oof_rmse = np.sqrt(mean_squared_error(y_wf.loc[oof_wf.index], oof_wf))

va_wf = wf_agg(va_rows, va_pred); te_wf = wf_agg(te_rows, te_pred)
valid_rmse = np.sqrt(mean_squared_error(valid_Ya.loc[va_wf.index], va_wf))
test_rmse  = np.sqrt(mean_squared_error(test_Ya.loc[te_wf.index],  te_wf))

print(f"CV OOF (WF) RMSE : {oof_rmse:.4f}")
print(f"Valid       RMSE : {valid_rmse:.4f}   (v3 최선 62.31)")
print(f"Test        RMSE : {test_rmse:.4f}   (v3 최선 60.51)")

CV OOF (WF) RMSE : 62.6337
Valid       RMSE : 61.3838   (v3 최선 62.31)
Test        RMSE : 60.5215   (v3 최선 60.51)


## Cell 7 — 피처 중요도 & 제출 저장

In [7]:
imp = pd.Series(m.booster_.feature_importance("gain"), index=USE).sort_values(ascending=False)
print(imp.head(20))
# row_pos / *_wf_* / C23_te 가 상위에 있으면 row-level·재탐색이 실제로 기여한 것

vs = valid_Yp.copy(); vs["C65"] = vs["C64"].map(va_wf); vs.to_csv(OUTPUT_DIR/"valid_Y_submit.csv", index=False)
ts = test_Yp.copy();  ts["C65"] = ts["C64"].map(te_wf); ts.to_csv(OUTPUT_DIR/"test_Y_submit.csv", index=False)
json.dump({"oof":round(oof_rmse,4),"valid":round(valid_rmse,4),"test":round(test_rmse,4)},
          open(OUTPUT_DIR/"results.json","w"), indent=2, ensure_ascii=False)
print("저장 완료")

C11_wf_min     5.479062e+10
C33            1.241924e+10
C12_wf_max     1.182452e+10
C33_wf_mean    4.557512e+09
C12_wf_min     8.943647e+08
C33_wf_min     6.688646e+08
C33_wf_max     4.833281e+08
C4_wf_mean     3.180791e+08
C12_wf_mean    2.731585e+08
C4_wf_std      1.713760e+08
C61_wf_min     1.508582e+08
C60_wf_std     1.209858e+08
C60_wf_mean    1.191926e+08
C52_wf_std     1.110588e+08
C27_wf_max     1.100412e+08
C25_wf_mean    1.025223e+08
C62_wf_min     1.020716e+08
C59_wf_mean    1.014333e+08
C27_wf_std     9.837071e+07
C25_wf_std     9.603568e+07
dtype: float64
저장 완료
